In [3]:
import numpy as np, pandas as pd, joblib
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor
import tensorflow as tf

# Load data from feature engineering notebook
df = pd.read_csv("../models/processed_sales_data.csv")
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df = df.dropna(subset=['Order_Date'])
df = df.sort_values('Order_Date').reset_index(drop=True)

# Recreate features & split
TARGET = 'Net_Sales_Value'
ML_FEATURES = [
    'Year','Month_Number','Quarter_Num','Week_Num',
    'Day_Num','Month_Sin','Month_Cos','Day_Sin','Day_Cos',
    'Is_Weekend',
    'Zone_enc','Tier_enc','City_enc','Beat_Name_enc',
    'DS_Type_enc','Channel_Type_enc','Outlet_Type_enc',
    'SBU_enc','Category_enc','SKU_Code_enc',
    'MRP','Distributor_Price','Cost_Price',
    'Discount_Pct','Gross_Margin_Pct','Achievement_Pct',
    'Distributor_Margin_Pct','Price_Realization_Pct',
    'Dist_Rank','SKU_Rank',
    'Payment_Mode_enc','Payment_Status_enc',
    'Delivery_Status_enc','Volume_Tier_enc',
    'Delivery_Delay_Days','Order_Processing_Days',
    'Sales_Lag_1','Sales_Lag_3','Sales_Lag_7',
    'Sales_Lag_14','Sales_Lag_21','Sales_Lag_30',
    'Rolling_3','Rolling_7','Rolling_14',
    'Rolling_21','Rolling_30',
    'Sales_Growth',
    'Target_Value','Target_Gap','Achievement_Ratio',
]
ML_FEATURES = [f for f in ML_FEATURES if f in df.columns]
X = df[ML_FEATURES]
y = df[TARGET]

split_idx = int(len(X) * 0.80)
X_test = X.iloc[split_idx:]
y_test = y.iloc[split_idx:]

# Load XGBoost model & predictions
model_xgb = joblib.load('../models/xgb_model.pkl')
scaler_X = joblib.load('../models/scaler_X.pkl')
X_test_sc = scaler_X.transform(X_test)
preds_xgb = model_xgb.predict(X_test_sc)

# Load LSTM scaler and recreate sequences for weekly aggregation
scaler_y = joblib.load('../models/scaler_y.pkl')
df_full = pd.read_csv("../Dataset.csv", parse_dates=['Order_Date'])
df_sales = df_full[df_full['Is_Return']=='No'].copy()
df_sales['Week_Start'] = (
    df_sales['Order_Date']
    - pd.to_timedelta(df_sales['Order_Date'].dt.weekday, unit='D')
)
weekly = (df_sales
          .groupby('Week_Start')['Net_Sales_Value']
          .sum().reset_index()
          .sort_values('Week_Start').reset_index(drop=True))

values = weekly['Net_Sales_Value'].values.reshape(-1,1)
scaled = scaler_y.fit_transform(values)

WINDOW = 8
X_seq, y_seq = [], []
for i in range(WINDOW, len(scaled)):
    X_seq.append(scaled[i-WINDOW:i, 0])
    y_seq.append(scaled[i, 0])

X_seq = np.array(X_seq).reshape(-1, WINDOW, 1)
y_seq = np.array(y_seq)

split_lstm = int(len(X_seq) * 0.80)
X_test_lstm = X_seq[split_lstm:]
y_test_lstm = y_seq[split_lstm:]

# Create simple LSTM baseline (or load if exists)
try:
    model_lstm = tf.keras.models.load_model('../models/lstm_model.h5')
    preds_lstm = model_lstm.predict(X_test_lstm, verbose=0).flatten()
except:
    # If LSTM model doesn't exist, use simple moving average as baseline
    print("LSTM model not found, using moving average baseline...")
    preds_lstm = np.array([np.mean(scaled[max(0, i-WINDOW):i]) for i in range(len(X_test_lstm))])
    
preds_lstm_scaled = preds_lstm.reshape(-1, 1) if preds_lstm.ndim == 1 else preds_lstm
actual_lstm = scaler_y.inverse_transform(y_test_lstm.reshape(-1, 1))
preds_lstm_inv = scaler_y.inverse_transform(preds_lstm_scaled).flatten()

print(f"✓ Models loaded | XGB preds: {len(preds_xgb)} | Weekly baseline: {len(preds_lstm_inv)}")

LSTM model not found, using moving average baseline...
✓ Models loaded | XGB preds: 3984 | Weekly baseline: 20


e:\Imarticus BNGLR\FMCG Sales Forecsting 24-25\venv\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
e:\Imarticus BNGLR\FMCG Sales Forecsting 24-25\venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [5]:
from sklearn.metrics import (mean_squared_error,
                              mean_absolute_error, r2_score)

# XGBoost operates on row-level data (transaction level)
# LSTM operates on weekly aggregated data
# To combine: aggregate XGBoost predictions to weekly too

# --- Align XGBoost to weekly level ---
df_test_rows = df.iloc[split_idx:].copy()
df_test_rows['Week_Start'] = (
    df_test_rows['Order_Date']
    - pd.to_timedelta(df_test_rows['Order_Date'].dt.weekday, unit='D')
)
df_test_rows['XGB_Pred'] = preds_xgb

xgb_weekly = (df_test_rows
    .groupby('Week_Start')['XGB_Pred']
    .sum().values)

# Match lengths and align predictions
min_len = min(len(xgb_weekly), len(preds_lstm_inv))
xgb_w   = xgb_weekly[:min_len]
lstm_w  = preds_lstm_inv[:min_len]
actual  = actual_lstm.flatten()[:min_len]

# Remove NaN values
mask = ~(np.isnan(actual) | np.isnan(xgb_w) | np.isnan(lstm_w))
xgb_w   = xgb_w[mask]
lstm_w  = lstm_w[mask]
actual  = actual[mask]

# Weighted hybrid: 60% XGBoost + 40% baseline/LSTM
W_XGB, W_LSTM = 0.60, 0.40
hybrid_pred = W_XGB * xgb_w + W_LSTM * lstm_w

rmse_hyb = np.sqrt(mean_squared_error(actual, hybrid_pred))
mae_hyb  = mean_absolute_error(actual, hybrid_pred)
r2_hyb   = r2_score(actual, hybrid_pred)

print(f"Hybrid RMSE : ₹{rmse_hyb:,.2f}")
print(f"Hybrid MAE  : ₹{mae_hyb:,.2f}")
print(f"Hybrid R²   : {r2_hyb:.4f}")

Hybrid RMSE : ₹132,284.07
Hybrid MAE  : ₹108,899.90
Hybrid R²   : -0.3594
